# Tutorial 3: Generation & Sampling

This tutorial covers loading trained TinyGPT weights and generating text with
different sampling strategies.

**Difficulty:** Beginner
**Estimated time:** 5 minutes

## 1. Load the trained model

In [ ]:
import sys
sys.path.insert(0, '../laboratory/python/lab_en')

from tiny_gpt_trainer import load_trained_model

# Use the pre-trained weights checked into the repo
WEIGHTS = "../laboratory/python/lab_en/results/models/tiny_gpt_trained.npz"
BPE     = "../laboratory/python/lab_en/results/models/tiny_gpt_bpe.json"

model, tokenizer = load_trained_model(WEIGHTS, BPE)
print(f"Loaded: {model.n_layers} layers, {model.n_params:,} params")
print(f"BPE: {tokenizer.n_merges} merges, vocab={tokenizer.vocab_size}")

## 2. Greedy decoding (temperature=0)

Greedy decoding picks the highest-probability token at every step. It's
deterministic but tends to get stuck in repetition loops.

In [ ]:
from tiny_gpt_trainer import generate_sample

prompt = "def train("
ids = tokenizer.encode(prompt)
out = generate_sample(model, ids, max_new_tokens=50, temperature=0.0)
print(tokenizer.decode(out))
# Typical output: 'def train(arararars.llllllll...)'
# The model has memorized that 'ar' is a frequent bigram and gets stuck.

## 3. Stochastic sampling (temperature=0.5)

Temperature > 0 introduces randomness. Higher temperature = more random.

In [ ]:
for _ in range(3):
    out = generate_sample(model, ids, max_new_tokens=50, temperature=0.5)
    print(tokenizer.decode(out))
    print("---")

## 4. Temperature sweep

Compare output at different temperatures. The model produces real Python
tokens (`Float`, `String`, `return`, `import`) mixed with random tokens.

In [ ]:
import numpy as np

temperatures = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5]
samples = {}

for temp in temperatures:
    np.random.seed(42)  # reproducible per-temperature
    out = generate_sample(model, ids, max_new_tokens=80, temperature=temp)
    samples[temp] = tokenizer.decode(out)

for temp, text in samples.items():
    print(f"\n=== T={temp} ===")
    print(text[:200])

## 5. Batch generation

Generate from multiple prompts to see how the model handles different contexts.

In [ ]:
prompts = [
    "def train(",
    "import numpy",
    "class Tiny",
    "The RMT-LLM",
    "# A comment",
]

for prompt in prompts:
    ids = tokenizer.encode(prompt)
    out = generate_sample(model, ids, max_new_tokens=40, temperature=0.7)
    text = tokenizer.decode(out)
    print(f"\nPrompt: {prompt!r}")
    print(f"  Out:  {text!r}")

## 6. Measuring generation speed

On a typical CPU you should see ~50-100 tokens/sec for the 12-layer
hidden-128 model.

In [ ]:
import time

n_tokens = 100
n_runs = 5

times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    out = generate_sample(model, ids, max_new_tokens=n_tokens, temperature=0.5)
    times.append(time.perf_counter() - t0)

mean_time = sum(times) / n_runs
tokens_per_sec = n_tokens / mean_time
print(f"Mean time: {mean_time:.3f}s for {n_tokens} tokens")
print(f"Throughput: {tokens_per_sec:.1f} tokens/sec")

## 7. Inspecting the attention pattern

The forward pass returns attention weights per layer per head. They should
show a **lower-triangular** pattern (causal masking).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Forward pass with cache
x = np.array(tokenizer.encode("def train("))
logits, cache = model.forward_with_cache(x)

# cache contains per-layer attention weights
layer_0_attn = cache["layers"][0]["attention_weights"]  # (n_heads, T, T)
print(f"Attention shape: {layer_0_attn.shape}")  # (4, 10, 10)

# Plot head 0
plt.figure(figsize=(8, 6))
plt.imshow(layer_0_attn[0], cmap='viridis')
plt.colorbar(label='Attention weight')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.title('Layer 0, Head 0 attention (causal mask visible)')
plt.show()

## 8. Comparing to the untrained model

The trained model produces real Python tokens; the untrained model produces
pure noise.

In [ ]:
from tiny_gpt import TinyGPT, TinyGPTConfig

# Build an untrained model with the same architecture
config = TinyGPTConfig(
    vocab_size=512, hidden_dim=128, n_layers=12, n_heads=4, max_seq_len=32
)
untrained = TinyGPT(config)
untrained.init_weights(seed=0)

# Compare outputs
prompt = "def train("
ids = tokenizer.encode(prompt)

trained_out = generate_sample(model, ids, max_new_tokens=30, temperature=0.5)
untrained_out = generate_sample(untrained, ids, max_new_tokens=30, temperature=0.5)

print("Trained:  ", tokenizer.decode(trained_out))
print("Untrained:", tokenizer.decode(untrained_out))

## What's next?

- [Tutorial 4: Multilingual Labs](04_multilingual_labs.ipynb)
- [API: TinyGPT](https://wild8highlander.github.io/rmt-llm-research/api/tinygpt/)
- [Architecture: Request Flow](https://wild8highlander.github.io/rmt-llm-research/architecture/request-flow/)